In [0]:
# ==============================================================================
# PIPELINE: Distributed Batch Inference via Pandas UDF
# DESCRIPTION: Loads the registered ResNet-50 model from Unity Catalog, defines
# a Pandas UDF that scores raw image bytes against the "prod" alias, runs
# distributed inference over all resized training images from the Silver layer,
# and writes the predicted damage labels and confidence scores to the Gold table.
# ==============================================================================

from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import StructType, StructField, StringType, FloatType
import pandas as pd

DAMAGE_PRED_SCHEMA = StructType([
    StructField("label", StringType()),
    StructField("score", FloatType()),
])

@pandas_udf(DAMAGE_PRED_SCHEMA)
def predict_damage_udf(content: pd.Series) -> pd.DataFrame:
    import mlflow
    model = mlflow.pyfunc.load_model("models:/dbr_dev.joshuandegwa_gold.claims_damage_level@prod")
    return model.predict(pd.DataFrame({"content": content}))

(
    spark.table("dbr_dev.joshuandegwa_silver.training_images_resized")
    .withColumn("damage_prediction", predict_damage_udf(col("content")))
    .write
    .mode("overwrite")
    .saveAsTable("dbr_dev.joshuandegwa_gold.damage_predictions")
)

print("Batch inference completed successfully and saved to dbr_dev.joshuandegwa_gold.damage_predictions!")

In [0]:
# Load and display the Gold predictions table
predictions = spark.table('dbr_dev.joshuandegwa_gold.damage_predictions')
display(predictions)

In [0]:
# 1. Load the Gold predictions table and flatten the damage_prediction struct
predictions = spark.table('dbr_dev.joshuandegwa_gold.damage_predictions')

results = (
    predictions
    .selectExpr("path", "label", "damage_prediction.label as predictions", "damage_prediction.score as score")
    .toPandas()
)

# 2. Display the resulting evaluation dataframe
display(results)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create confusion matrix
conf_matrix = pd.crosstab(results['label'], results['predictions'])

# Plot confusion matrix with labels and clean sizing
plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, cmap="Blues", fmt='d', cbar=False)

plt.title("Claims Damage Classification — Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import when, lit

CONFIDENCE_THRESHOLD = 0.5

# Identify images that are misclassified OR have low confidence
all_results = (
    predictions
    .selectExpr("path", "label", "damage_prediction.label as predicted", "damage_prediction.score as confidence")
)

for_review = (
    all_results
    .filter(f"label != predicted OR confidence < {CONFIDENCE_THRESHOLD}")
    .withColumn("reason", 
        when(col("label") != col("predicted"), lit("misclassified"))
        .otherwise(lit("low_confidence"))
    )
    .orderBy("confidence")
)

review_count = for_review.count()
misclassified_count = for_review.filter(col("reason") == "misclassified").count()
low_conf_count = for_review.filter(col("reason") == "low_confidence").count()

print(f"Total flagged for review: {review_count}")
print(f"  Misclassified: {misclassified_count}")
print(f"  Low confidence (< {CONFIDENCE_THRESHOLD}): {low_conf_count}")
display(for_review)

# Save to review table for human inspection
for_review.write.mode("overwrite").saveAsTable("dbr_dev.joshuandegwa_gold.training_images_for_review")
print(f"Saved {review_count} images to dbr_dev.joshuandegwa_gold.training_images_for_review")